In [1]:
# ============================================
# OpenSky Data Cleaning
# MSc Dissertation Project
# ============================================

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# ============================================
# Load Raw Dataset
# ============================================

df = pd.read_csv("../data/raw/opensky/opensky_data_uncleaned.csv")

print("Dataset loaded successfully.")
print(df.shape)

Dataset loaded successfully.
(7561, 16)


In [3]:
# ============================================
# Create Working Copy
# ============================================

clean_df = df.copy()

print("Working copy created.")

Working copy created.


In [4]:
# ============================================
# Missing Values Before Cleaning
# ============================================

missing_before = pd.DataFrame({
    "Missing Values": clean_df.isnull().sum(),
    "Percentage (%)": (
        clean_df.isnull().sum() /
        len(clean_df) * 100
    ).round(2)
})

display(missing_before)

,Missing Values,Percentage (%)
icao24,0,0.00
callsign,117,1.55
origin_country,0,0.00
time_position,45,0.60
last_constant,0,0.00
longitude,45,0.60
latitude,45,0.60
baro_altitude,724,9.58
on_ground,0,0.00
velocity,0,0.00


In [5]:
# ============================================
# Remove Records Missing Position Information
# ============================================

rows_before = len(clean_df)

clean_df = clean_df.dropna(
    subset=[
        "time_position",
        "latitude",
        "longitude"
    ]
)

rows_after = len(clean_df)

print(f"Rows before cleaning : {rows_before}")
print(f"Rows after cleaning  : {rows_after}")
print(f"Rows removed         : {rows_before - rows_after}")

Rows before cleaning : 7561
Rows after cleaning  : 7516
Rows removed         : 45


## 3.1 Handling Missing Callsigns

A small number of aircraft observations contain missing flight callsigns. Since the aircraft identifier (`icao24`) remains available and the observations retain valid positional and operational information, these records are preserved. Missing callsigns are replaced with the value `"UNKNOWN"` to maintain data consistency while indicating that the flight identifier was unavailable.

In [6]:
# ============================================
# Handle Missing Callsigns
# ============================================

clean_df["callsign"] = clean_df["callsign"].fillna("UNKNOWN")

print("Missing Callsigns:",
      clean_df["callsign"].isnull().sum())

Missing Callsigns: 0


## 3.2 Handling Missing Barometric Altitude

The `baro_altitude` attribute contains approximately 9.6% missing values. Removing these observations would unnecessarily reduce the dataset size and may negatively affect subsequent analyses. Since the altitude distribution is positively skewed, the missing values are replaced using the median rather than the mean, as the median is less sensitive to extreme observations.

In [7]:
# ============================================
# Handle Missing Barometric Altitude
# ============================================

median_baro = clean_df["baro_altitude"].median()

clean_df["baro_altitude"] = clean_df[
    "baro_altitude"
].fillna(median_baro)

print(
    "Remaining Missing Barometric Altitude:",
    clean_df["baro_altitude"].isnull().sum()
)

Remaining Missing Barometric Altitude: 0


## 3.3 Handling Missing Vertical Rate

The `vertical_rate` variable represents the rate of climb or descent. Missing values were replaced using the median value to preserve the overall distribution while avoiding the influence of extreme climb or descent rates.

In [8]:
# ============================================
# Handle Missing Vertical Rate
# ============================================

median_vertical = clean_df[
    "vertical_rate"
].median()

clean_df["vertical_rate"] = clean_df[
    "vertical_rate"
].fillna(median_vertical)

print(
    "Remaining Missing Vertical Rate:",
    clean_df["vertical_rate"].isnull().sum()
)

Remaining Missing Vertical Rate: 0


## 3.4 Handling Missing Geometric Altitude

The `geo_altitude` attribute is an important feature for flight dynamics analysis. Missing observations were replaced using the median value to retain the maximum number of aircraft observations while preserving the central tendency of the dataset.

In [9]:
# ============================================
# Handle Missing Geometric Altitude
# ============================================

median_geo = clean_df[
    "geo_altitude"
].median()

clean_df["geo_altitude"] = clean_df[
    "geo_altitude"
].fillna(median_geo)

print(
    "Remaining Missing Geo Altitude:",
    clean_df["geo_altitude"].isnull().sum()
)

Remaining Missing Geo Altitude: 0


## 3.5 Cleaning Text Attributes

Text-based attributes were standardized by removing leading and trailing whitespace and converting identifiers to uppercase where appropriate. This improves consistency during grouping, filtering, and dataset integration while preserving the original semantic meaning of the data.

In [10]:
# ============================================
# Clean Text-Based Columns
# ============================================

# Remove leading/trailing spaces
clean_df["icao24"] = clean_df["icao24"].astype(str).str.strip()

clean_df["callsign"] = clean_df["callsign"].astype(str).str.strip()

clean_df["origin_country"] = (
    clean_df["origin_country"]
    .astype(str)
    .str.strip()
)

# Convert identifiers to uppercase
clean_df["icao24"] = clean_df["icao24"].str.upper()

clean_df["callsign"] = clean_df["callsign"].str.upper()

print("Text fields cleaned successfully.")

Text fields cleaned successfully.


In [11]:
# ============================================
# Final Missing Value Check
# ============================================

print("=" * 60)
print("Missing Values After Cleaning")
print("=" * 60)

display(clean_df.isnull().sum())

Missing Values After Cleaning


icao24                0
callsign              0
origin_country        0
time_position         0
last_constant         0
longitude             0
latitude              0
baro_altitude         0
on_ground             0
velocity              0
true_track            0
vertical_rate         0
geo_altitude          0
squawk             4286
spi                   0
position_source       0
dtype: int64

In [12]:
# ============================================
# Final Dataset Information
# ============================================

print("=" * 60)
print("Cleaned Dataset Information")
print("=" * 60)

clean_df.info()

Cleaned Dataset Information
<class 'pandas.core.frame.DataFrame'>
Index: 7516 entries, 0 to 7560
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   icao24           7516 non-null   object 
 1   callsign         7516 non-null   object 
 2   origin_country   7516 non-null   object 
 3   time_position    7516 non-null   float64
 4   last_constant    7516 non-null   int64  
 5   longitude        7516 non-null   float64
 6   latitude         7516 non-null   float64
 7   baro_altitude    7516 non-null   float64
 8   on_ground        7516 non-null   bool   
 9   velocity         7516 non-null   float64
 10  true_track       7516 non-null   float64
 11  vertical_rate    7516 non-null   float64
 12  geo_altitude     7516 non-null   float64
 13  squawk           3230 non-null   float64
 14  spi              7516 non-null   bool   
 15  position_source  7516 non-null   int64  
dtypes: bool(2), float64(9), int64(2), obj

In [13]:
# ============================================
# Dataset Dimensions After Cleaning
# ============================================

print("=" * 60)
print("Dataset Dimensions")
print("=" * 60)

print(f"Rows    : {clean_df.shape[0]}")
print(f"Columns : {clean_df.shape[1]}")

Dataset Dimensions
Rows    : 7516
Columns : 16


In [14]:
# ============================================
# Save Cleaned Dataset
# ============================================

clean_df.to_csv(
    "../data/processed/opensky_processed.csv",
    index=False
)

print("✅ Cleaned OpenSky dataset saved successfully.")
print("Location: data/processed/opensky_processed.csv")

✅ Cleaned OpenSky dataset saved successfully.
Location: data/processed/opensky_processed.csv


## 3.6 Handling Missing Squawk Values

The `squawk` attribute contains a high proportion of missing values (approximately 57%). Unlike continuous numerical variables, the squawk code is a categorical transponder identifier assigned by Air Traffic Control and does not represent a measurable operational parameter.

Since squawk codes are not used as predictive features in the proposed anomaly detection models or hybrid risk assessment framework, imputing missing values would introduce artificial identifiers without operational meaning. Removing all observations containing missing squawk values would result in a substantial loss of data.

Therefore, the missing squawk values were retained without imputation. The attribute will remain available for descriptive analysis but will not be used as an input feature for machine learning.

In [15]:
# ============================================
# Reset DataFrame Index
# ============================================

clean_df.reset_index(drop=True, inplace=True)

print("Index has been reset successfully.")

Index has been reset successfully.


In [16]:
clean_df.head()

,icao24,callsign,origin_country,time_position,last_constant,longitude,latitude,baro_altitude,on_ground,velocity,true_track,vertical_rate,geo_altitude,squawk,spi,position_source
0,80162D,AXB848,India,1.767745e+09,1767745048,52.7391,25.4425,9608.82,False,257.34,119.59,5.20,9829.80,3112.0,False,0
1,AE1FA0,72209,United States,1.767745e+09,1767745047,-84.9380,38.1463,571.50,False,73.38,12.96,0.33,487.68,NaN,False,0
2,AC96B8,AAL1175,United States,1.767745e+09,1767745048,-102.0238,34.0962,10363.20,False,196.90,291.13,0.00,10683.24,NaN,False,0
3,C81BD3,ZKAMK,New Zealand,1.767745e+09,1767744917,174.5408,-35.8060,624.84,False,56.90,165.34,0.65,670.56,NaN,False,0
4,AA56DB,UAL2447,United States,1.767745e+09,1767745048,-102.7402,37.4696,10972.80,False,200.08,327.49,0.00,11155.68,NaN,False,0


In [17]:
clean_df.to_csv(
    "../data/processed/opensky_processed.csv",
    index=False
)